## IEEE-CIS 사기 거래 탐지 데이터

In [ ]:
from pathlib import Path
from zipfile import ZipFile, is_zipfile
import subprocess
import sys

import pandas as pd
from IPython.display import display

DATA_DIR = Path("data")

### IEEE-CIS 데이터

IEEE-CIS Fraud Detection은 Vesta의 전자상거래 데이터다. `train_transaction.csv`는 총 590,540행·394개 열로 구성되며, 거래별 정보와 사기 여부를 담고 있다. 한 행은 거래 한 건이다.

출처: Kaggle, IEEE-CIS Fraud Detection / Vesta. 이용 조건은 해당 대회의 규정을 따른다.

### 항목 설명

| 열 | 개수 | 의미 |
| --- | --- | --- |
| `TransactionID` | 1 | 거래 식별자 |
| `isFraud` | 1 | 사기 여부, 0=정상·1=사기 |
| `TransactionDT` | 1 | 기준 시점부터의 경과 시간 |
| `TransactionAmt` | 1 | 거래 금액 |
| `ProductCD` | 1 | 상품 코드 |
| `card1`~`card6` | 6 | 카드 관련 정보 |
| `addr1`, `addr2` | 2 | 주소 관련 정보 |
| `dist1`, `dist2` | 2 | 거리 관련 정보 |
| `P_emaildomain`, `R_emaildomain` | 2 | 구매자·수취인의 이메일 도메인 |
| `C1`~`C14` | 14 | 횟수·개수 관련 특성 |
| `D1`~`D15` | 15 | 시간 간격 관련 특성 |
| `M1`~`M9` | 9 | 정보 일치 여부 관련 특성 |
| `V1`~`V339` | 339 | Vesta가 만든 순위·횟수·정보 간 관계 등의 가공 특성 |

익명화된 열의 개별 의미와 계산 방식은 공개된 설명 범위 안에서 해석한다. `TransactionDT`는 상대 시간이다.

### 직접 다운로드하는 경우

노트북에서 사용하는 Python 환경에 `pip install kaggle`로 Kaggle 패키지를 설치한다. Kaggle 설정에서 API 토큰을 발급받아 사용자 홈 폴더의 `.kaggle/access_token` 파일에 저장하고, IEEE-CIS 대회 규칙에 동의한다.

`DATA_SOURCE = "kaggle"`로 설정하면 `train_transaction.csv`를 `data` 폴더에 내려받는다.

### 주어진 ZIP 파일을 사용하는 경우

[Google Drive에서 데이터 파일을 내려받아](https://drive.google.com/file/d/1p7dXQT8fdqNjNcl2rKGfoIDrYNngtCZM/view?usp=drive_link) 노트북과 같은 위치의 `data` 폴더에 `ieee-fraud-detection.zip`으로 저장한다. ZIP 파일은 압축을 풀지 않고, `DATA_SOURCE = "provided_zip"`으로 설정한다.

In [2]:
DATA_SOURCE = "provided_zip"  # "kaggle" 또는 "provided_zip"
PROVIDED_ZIP_PATH = DATA_DIR / "ieee-fraud-detection.zip"
KAGGLE_DOWNLOAD_DIR = DATA_DIR

In [ ]:
fraud_path = None

if DATA_SOURCE == "provided_zip":
    fraud_path = Path(PROVIDED_ZIP_PATH)
    if not fraud_path.is_file():
        raise FileNotFoundError(f"주어진 ZIP 파일을 찾을 수 없습니다. PROVIDED_ZIP_PATH를 확인하세요: {fraud_path}")
    if not is_zipfile(fraud_path):
        raise ValueError(f"올바른 ZIP 파일이 아닙니다: {fraud_path}")
elif DATA_SOURCE == "kaggle":
    candidates = [
        KAGGLE_DOWNLOAD_DIR / "train_transaction.csv",
        KAGGLE_DOWNLOAD_DIR / "train_transaction.csv.zip",
    ]
    fraud_path = next((path for path in candidates if path.is_file()), None)
    if fraud_path is None:
        # 현재 커널의 Python으로 Kaggle CLI를 실행한다. -f는 파일, -p는 저장 폴더다.
        subprocess.run([
            sys.executable, "-m", "kaggle", "competitions", "download",
            "ieee-fraud-detection", "-f", "train_transaction.csv",
            "-p", str(KAGGLE_DOWNLOAD_DIR),
        ], check=True)
        fraud_path = next((path for path in candidates if path.is_file()), None)
        if fraud_path is None:
            raise FileNotFoundError("다운로드 결과에서 train_transaction.csv 또는 train_transaction.csv.zip을 찾을 수 없습니다.")
else:
    raise ValueError('DATA_SOURCE는 "kaggle" 또는 "provided_zip"으로 설정하세요.')

### 거래 데이터 불러오기

In [4]:
PREVIEW_ROWS = 100_000  # 읽을 행 수. None이면 전체 행을 읽는다.

if is_zipfile(fraud_path):
    with ZipFile(fraud_path) as archive:
        with archive.open("train_transaction.csv") as source:
            fraud_preview = pd.read_csv(source, nrows=PREVIEW_ROWS)
else:
    fraud_preview = pd.read_csv(fraud_path, nrows=PREVIEW_ROWS)

print("불러온 행·열:", fraud_preview.shape)
display(fraud_preview.head(10))

불러온 행·열: (100000, 394)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2987005,0,86510,49.0,W,5937,555.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2987006,0,86522,159.0,W,12308,360.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2987007,0,86529,422.5,W,12695,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2987008,0,86535,15.0,H,2803,100.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,2987009,0,86536,117.0,W,17399,111.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
